<a href="https://colab.research.google.com/github/NyxLumen/Encephlo/blob/main/Notebooks/VIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate scikit-learn

In [3]:
from google.colab import drive
import zipfile
import os

# 1. Mount Google Drive
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define Paths
zip_path = "/content/drive/My Drive/mri_train.zip"
extract_path = "/content/dataset"

# 3. Verify & Extract
if os.path.exists(zip_path):
    print(f"✅ Found zip file at: {zip_path}")
    print(f"📂 Extracting to: {extract_path} ...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("🎉 SUCCESS! Data is ready.")
    print(f"   - Training folder: {extract_path}/Training (hopefully!)")
else:
    print(f"❌ ERROR: Could not find '{zip_path}'")
    print("   - Did you name it exactly 'mri_train.zip'?")
    print("   - Is it in the main folder of your Google Drive?")

🔄 Mounting Google Drive...
Mounted at /content/drive
✅ Found zip file at: /content/drive/My Drive/mri_train.zip
📂 Extracting to: /content/dataset ...
🎉 SUCCESS! Data is ready.
   - Training folder: /content/dataset/Training (hopefully!)


In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm.auto import tqdm

# ─────────────────────────────────────────────
# 1. SETUP & PATH CHECK
# ─────────────────────────────────────────────
# We look for the 'Training' folder inside the extracted data
BASE_DIR = "/content/dataset"
DATA_DIR = BASE_DIR

# If the zip had a 'Training' subfolder, use it. Otherwise, use base.
if os.path.exists(os.path.join(BASE_DIR, "Training")):
    DATA_DIR = os.path.join(BASE_DIR, "Training")

print(f"📂 Scanning directory: {DATA_DIR}")
print(f"   Contents: {os.listdir(DATA_DIR)[:5]}...") # Show first 5 files/folders

OUTPUT_MODEL_PATH = "/content/vit_feature_extractor.pt"
BATCH_SIZE = 16
EPOCHS = 8
LR = 2e-5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Hardware Accelerator: {device}")

# ─────────────────────────────────────────────
# 2. DATASET CLASS
# ─────────────────────────────────────────────
class MRIDataset(Dataset):
    def __init__(self, filepaths, labels, processor):
        self.filepaths = filepaths
        self.labels = labels
        self.processor = processor

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        image = Image.open(img_path).convert("RGB")
        encoding = self.processor(image, return_tensors="pt")
        return {
            'pixel_values': encoding.pixel_values.squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def get_file_list(directory):
    classes = sorted([d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))])
    class_indices = {name: idx for idx, name in enumerate(classes)}
    filepaths = []
    labels = []

    for class_name in classes:
        class_dir = os.path.join(directory, class_name)
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepaths.append(os.path.join(class_dir, fname))
                labels.append(class_indices[class_name])
    return filepaths, labels, class_indices

# ─────────────────────────────────────────────
# 3. PREPARE DATA
# ─────────────────────────────────────────────
filepaths, labels, class_indices = get_file_list(DATA_DIR)
print(f"✅ Found {len(filepaths)} images. Classes: {class_indices}")

if len(filepaths) == 0:
    raise ValueError("❌ No images found! Check the path above.")

# Split: 85% Train, 15% Validation
train_paths, val_paths, train_labels, val_labels = train_test_split(
    filepaths, labels, test_size=0.15, random_state=42, stratify=labels
)

# Load Processor
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")
train_ds = MRIDataset(train_paths, train_labels, processor)
val_ds = MRIDataset(val_paths, val_labels, processor)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ─────────────────────────────────────────────
# 4. TRAIN MODEL
# ─────────────────────────────────────────────
print("\n⬇️ Downloading ViT Model...")
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=len(class_indices)
)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("\n🏋️ STARTING TRAINING...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        loop.set_description(f"Epoch {epoch+1}/{EPOCHS}")
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Avg Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

# ─────────────────────────────────────────────
# 5. SAVE
# ─────────────────────────────────────────────
print("\n💾 Saving Feature Extractor...")
torch.save(model.vit.state_dict(), OUTPUT_MODEL_PATH)
print(f"✅ DONE! Download {OUTPUT_MODEL_PATH} from the files tab.")

📂 Scanning directory: /content/dataset/Training
   Contents: ['pituitary', 'glioma', 'notumor', 'meningioma']...
🚀 Hardware Accelerator: cuda
✅ Found 5712 images. Classes: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]


⬇️ Downloading ViT Model...


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

...

Epoch 6: Avg Loss: 0.0177 | Acc: 0.9986


  0%|          | 0/304 [00:00<?, ?it/s]

Epoch 7: Avg Loss: 0.0144 | Acc: 0.9984


  0%|          | 0/304 [00:00<?, ?it/s]

Epoch 8: Avg Loss: 0.0193 | Acc: 0.9963

💾 Saving Feature Extractor...
✅ DONE! Download /content/vit_feature_extractor.pt from the files tab.


In [5]:
# ─────────────────────────────────────────────
# FINAL EXAM: VALIDATION CHECK
# ─────────────────────────────────────────────
model.eval() # Switch to evaluation mode (turns off dropout/batchnorm updates)
val_correct = 0
val_total = 0

print("👨‍⚕️ Running final diagnostic on Validation Set...")

with torch.no_grad(): # Disable gradient calculation (saves RAM)
    for batch in tqdm(val_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values)
        predictions = torch.argmax(outputs.logits, dim=1)

        val_correct += (predictions == labels).sum().item()
        val_total += labels.size(0)

val_acc = val_correct / val_total
print(f"\n🏆 Final Validation Accuracy: {val_acc*100:.2f}%")

if val_acc > 0.90:
    print("✅ STATUS: HOSPITAL-GRADE. Ready for deployment.")
else:
    print("⚠️ STATUS: DECENT. Good enough for a prototype.")

👨‍⚕️ Running final diagnostic on Validation Set...


  0%|          | 0/54 [00:00<?, ?it/s]


🏆 Final Validation Accuracy: 99.53%
✅ STATUS: HOSPITAL-GRADE. Ready for deployment.
